# EN4554 Assignment 2 — Task 1: Backpropagation

**Index number:** 220514C

**GitHub:** https://github.com/chamudiR/EN4554-Assignment-2-on-CNNs

<a href="https://colab.research.google.com/github/chamudiR/EN4554-Assignment-2-on-CNNs/blob/main/Task1_Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 0. Setup

* Fixed random seeds (Python, NumPy, PyTorch).
* CIFAR-10 training set split into **45 000 train / 5 000 validation** images with a seeded permutation.
* Per-channel normalisation uses the mean and standard deviation of the **45k training split only**, so no validation information leaks into preprocessing.
* The **test set is not loaded at all** in this task. It is used only once, in Task 4.

In [ ]:
import math, random, time
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device,
      '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')

In [ ]:
# ---- Load CIFAR-10 training set and make the 45k / 5k split ----
train_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)

X_all = torch.from_numpy(train_full.data).float().div(255).permute(0, 3, 1, 2)  
# (50000, 3, 32, 32) in [0, 1]
# permute(0, 3, 1, 2) reorders the axes to (N, C, H, W) = (50000, 3, 32, 32). This channels-first layout is what PyTorch conv layers expect.

y_all = torch.tensor(train_full.targets, dtype=torch.long)
CLASSES = train_full.classes

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45000], perm[45000:]
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val,   y_val   = X_all[val_idx],   y_all[val_idx]

# ---- Per-channel normalisation with TRAINING statistics only ----
mean = X_train.mean(dim=(0, 2, 3))
std  = X_train.std(dim=(0, 2, 3))
X_train = (X_train - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)
X_val   = (X_val   - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)

# ---- Flattened versions for the MLP (kept on the GPU: 45k x 3072 floats ~ 0.55 GB) ----
Xtr = X_train.reshape(len(X_train), -1).to(device)
Xva = X_val.reshape(len(X_val), -1).to(device)
ytr, yva = y_train.to(device), y_val.to(device)

print('train', tuple(Xtr.shape), '| val', tuple(Xva.shape))
print('channel mean (train):', mean.numpy().round(4), '| channel std (train):', std.numpy().round(4))
print('val class counts:', torch.bincount(y_val, minlength=10).tolist())

## Task 1(a): Layers with explicit forward / backward

Every layer stores what it needs during `forward()` and uses it in `backward()`. `backward()` takes $\partial L/\partial(\text{output})$ and returns $\partial L/\partial(\text{input})$. Parameter gradients are stored on the layer. Nothing here uses autograd: the parameter tensors have `requires_grad=False`, so PyTorch builds no graph.

For a mini-batch $X \in \mathbb{R}^{N\times d}$:

| Layer | Forward | Backward |
|---|---|---|
| Linear | $Z = XW + \mathbf{1}b^\top$ | $\frac{\partial L}{\partial W} = X^\top G,\quad \frac{\partial L}{\partial b} = \sum_n G_{n,:},\quad \frac{\partial L}{\partial X} = G W^\top$ |
| ReLU | $H = \max(0, Z)$ | $\frac{\partial L}{\partial Z} = G \odot \mathbb{1}[Z>0]$ |
| Sigmoid (for 1d) | $H = \sigma(Z)$ | $\frac{\partial L}{\partial Z} = G \odot H \odot (1-H)$ |
| Softmax + CE | $p = \mathrm{softmax}(z),\ L = -\frac1N\sum_n \log p_{n,y_n}$ | $\frac{\partial L}{\partial z} = \frac1N (p - \text{onehot}(y))$ |

Here $G$ is the gradient arriving from the layer above. Softmax and cross-entropy are fused because the combined gradient $p - \text{onehot}(y)$ is simple and numerically stable. The logits are shifted by their row maximum before exponentiating, which does not change the softmax but prevents overflow.

Initialisation:
* **Kaiming/He:** $W \sim \mathcal N(0, 2/n_\text{in})$. This keeps activation variance constant through ReLU layers.
* **Xavier/Glorot:** $W \sim \mathcal N(0, 2/(n_\text{in}+n_\text{out}))$. This was designed for symmetric activations such as tanh and sigmoid.

In [ ]:
class Linear:
    def __init__(self, in_features, out_features, init='kaiming', dtype=torch.float32, device='cpu'):
        if init == 'kaiming':
            std = math.sqrt(2.0 / in_features)
        elif init == 'xavier':
            std = math.sqrt(2.0 / (in_features + out_features))
        else:
            raise ValueError(init)
        # created on the CPU so that the same seed gives the same weights on any device
        self.W = (torch.randn(in_features, out_features, dtype=dtype) * std).to(device)
        self.b = torch.zeros(out_features, dtype=dtype, device=device)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x                                  # cache input for backward
        return x @ self.W + self.b

    def backward(self, grad_out):
        self.dW = self.x.T @ grad_out               # (in, N) @ (N, out)
        self.db = grad_out.sum(dim=0)
        return grad_out @ self.W.T                  # dL/dx, (N, out) @ (out, in)

    def params_and_grads(self):
        return [(self.W, self.dW), (self.b, self.db)]


class ReLU:
    def forward(self, x):
        self.mask = (x > 0).to(x.dtype)
        return x * self.mask

    def backward(self, grad_out):
        return grad_out * self.mask

    def params_and_grads(self):
        return []


class Sigmoid:
    def forward(self, x):
        self.out = torch.sigmoid(x)                 # elementwise 1/(1+e^-x); no autograd involved
        return self.out

    def backward(self, grad_out):
        return grad_out * self.out * (1.0 - self.out)

    def params_and_grads(self):
        return []


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values          # stability shift
        log_probs = z - torch.log(torch.exp(z).sum(dim=1, keepdim=True))
        self.probs = torch.exp(log_probs)
        self.y = y
        n = logits.shape[0]
        return -log_probs[torch.arange(n, device=logits.device), y].mean()

    def backward(self):
        n = self.probs.shape[0]
        grad = self.probs.clone()
        grad[torch.arange(n, device=grad.device), self.y] -= 1.0
        return grad / n

In [ ]:
class MLP:
    # sizes = [in, hidden_1, ..., hidden_k, out]; an activation follows every Linear except the last.
    def __init__(self, sizes, activation='relu', init='kaiming', dtype=torch.float32, device='cpu'):
        act = {'relu': ReLU, 'sigmoid': Sigmoid}[activation]
        self.layers = []
        for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(n_in, n_out, init, dtype, device))
            if i < len(sizes) - 2:
                self.layers.append(act())
        self.loss_fn = SoftmaxCrossEntropy()

    @property
    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad                                  # dL/d(input)

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def loss_and_grads(self, x, y):
        # one forward + backward pass; returns (loss, logits, dL/dx); parameter grads are left on the layers
        logits = self.forward(x)
        loss = self.loss_fn.forward(logits, y)
        dx = self.backward(self.loss_fn.backward())
        return loss, logits, dx

    def params_and_grads(self):
        return [pg for l in self.layers for pg in l.params_and_grads()]

    def num_params(self):
        return sum(p.numel() for p, _ in self.params_and_grads())


net = MLP([3072, 512, 256, 10])
print(' -> '.join(type(l).__name__ for l in net.layers))
print('parameters:', net.num_params())

## Task 1(b): Gradient checking

The manual gradients are compared with two independent references:

1. **Autograd.** The same forward pass is rebuilt from copies of the parameters using `torch` ops (`@`, `torch.relu`/`torch.sigmoid`, `F.cross_entropy`), and `loss.backward()` is called.
2. **Centred finite differences.** For every parameter entry, $\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + h) - L(\theta_i - h)}{2h}$. The truncation error is $O(h^2)$.

The relative error of each tensor is $\dfrac{\lVert g_\text{manual} - g_\text{ref}\rVert_2}{\max(\lVert g_\text{manual}\rVert_2, \lVert g_\text{ref}\rVert_2)}$.

All checks use **float64**. In float32 the round-off in the finite difference, about $\epsilon_\text{mach}\,|L|/h \approx 10^{-7}/10^{-3}$, would dominate and give relative errors around $10^{-3}$–$10^{-4}$ even for correct code.

The checks are:
* **Full check.** Every parameter and the input gradient are checked on a small MLP (32→24→16→10, batch of 8 random inputs), once with ReLU and once with sigmoid.
* **Full-size check.** The real 3072→512→256→10 network is checked on a batch of 64 CIFAR-10 images. It is compared with autograd on every entry, and with finite differences on 40 randomly chosen entries per tensor (checking all 1.7 M entries would need 3.4 M forward passes).

In [ ]:
def rel_error(a, b):
    return ((a - b).norm() / max(a.norm(), b.norm(), 1e-30)).item()


def autograd_grads(mlp, x, y):
    # Rebuild the same computation with autograd-tracked copies of the parameters.
    params = [p.detach().clone().requires_grad_(True) for p, _ in mlp.params_and_grads()]
    x_ag = x.detach().clone().requires_grad_(True)
    h, i = x_ag, 0
    for layer in mlp.layers:
        if isinstance(layer, Linear):
            h = h @ params[i] + params[i + 1]
            i += 2
        elif isinstance(layer, ReLU):
            h = torch.relu(h)
        elif isinstance(layer, Sigmoid):
            h = torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    return [p.grad for p in params], x_ag.grad


def fd_grad(mlp, x, y, tensor, idx=None, h=1e-6):
    # Centred finite differences of the loss w.r.t. `tensor` (modified in place, then restored).
    # idx: flat indices to check (None = all entries). Returns the estimates at those indices.
    flat = tensor.view(-1)
    idx = range(flat.numel()) if idx is None else idx
    out = []
    for i in idx:
        old = flat[i].item()
        flat[i] = old + h; lp = mlp.loss(x, y).item()
        flat[i] = old - h; lm = mlp.loss(x, y).item()
        flat[i] = old
        out.append((lp - lm) / (2 * h))
    return torch.tensor(out, dtype=tensor.dtype)


def param_names(mlp):
    return [f'{n}{k + 1}' for k in range(len(mlp.linears)) for n in ('W', 'b')]

In [ ]:
def full_check(activation, sizes=(32, 24, 16, 10), batch=8):
    set_seed(0)
    mlp = MLP(list(sizes), activation=activation, dtype=torch.float64)
    x = torch.randn(batch, sizes[0], dtype=torch.float64)
    y = torch.randint(0, sizes[-1], (batch,))

    _, _, dx = mlp.loss_and_grads(x, y)
    manual = [g.clone() for _, g in mlp.params_and_grads()] + [dx]
    ag_p, ag_x = autograd_grads(mlp, x, y)
    autograd = ag_p + [ag_x]
    fd = [fd_grad(mlp, x, y, p).view_as(p) for p, _ in mlp.params_and_grads()] + [fd_grad(mlp, x, y, x).view_as(x)]

    print(f'--- {activation.upper()} MLP {sizes}, all entries checked ---')
    print(f'{"tensor":>7} {"shape":>10} | {"rel err vs autograd":>20} {"allclose":>8} | {"rel err vs FD":>14} {"allclose":>8}')
    for name, m, a, f in zip(param_names(mlp) + ['x'], manual, autograd, fd):
        print(f'{name:>7} {str(tuple(m.shape)):>10} | {rel_error(m, a):20.2e} {str(torch.allclose(m, a)):>8} | '
              f'{rel_error(m, f):14.2e} {str(torch.allclose(m, f, rtol=1e-5, atol=1e-8)):>8}')
    print()

full_check('relu')
full_check('sigmoid')

In [ ]:
# Full-size network on real CIFAR-10 images
set_seed(1)
big = MLP([3072, 512, 256, 10], activation='relu', dtype=torch.float64)
xb = X_train[:64].reshape(64, -1).double()
yb = y_train[:64]

big.loss_and_grads(xb, yb)
manual = [g.clone() for _, g in big.params_and_grads()]
ag, _ = autograd_grads(big, xb, yb)

print(f'{"tensor":>7} {"shape":>12} | {"rel err vs autograd":>20} {"allclose":>8} | {"rel err vs FD (40 entries)":>26} {"allclose":>8}')
gen = torch.Generator().manual_seed(0)
for name, (p, _), m, a in zip(param_names(big), big.params_and_grads(), manual, ag):
    idx = torch.randperm(p.numel(), generator=gen)[:40].tolist()
    f = fd_grad(big, xb, yb, p, idx)
    m_sub = m.view(-1)[idx]
    print(f'{name:>7} {str(tuple(p.shape)):>12} | {rel_error(m, a):20.2e} {str(torch.allclose(m, a)):>8} | '
          f'{rel_error(m_sub, f):26.2e} {str(torch.allclose(m_sub, f, rtol=1e-5, atol=1e-8)):>8}')

**Interpretation.**

* **Manual vs autograd.** The relative errors are around $10^{-16}$, which is float64 machine precision, and `torch.allclose` returns `True` for every tensor. The manual backward pass computes the same quantities as autograd up to floating-point reordering.
* **Manual vs finite differences.** The relative errors are about $10^{-9}$–$10^{-11}$, and `allclose` is again `True` everywhere. Finite differences cannot reach $10^{-16}$: with $h = 10^{-6}$ the round-off term $\epsilon_\text{mach}|L|/h \approx 10^{-10}$ and the $O(h^2)$ truncation error set the floor. Errors below about $10^{-7}$ are the usual sign of a correct gradient. A bug, such as a missing $1/N$ or a transposed matrix, would give errors of order $10^{-1}$–$10^{0}$.
* **Two references.** Finite differences test the maths independently of PyTorch. Autograd tests the code at full precision. Agreement with both confirms the Linear, ReLU, Sigmoid and softmax cross-entropy backward passes, including $\partial L/\partial x$.
* **ReLU kink.** The finite-difference check could fail at a ReLU input that lies within $h$ of 0. No tensor failed here, so no pre-activation was that close to 0.

## Task 1(c): Training the MLP with mini-batch SGD on flattened CIFAR-10

* **Model:** 3072 → 512 → 256 → 10, ReLU, He initialisation (about 1.7 M parameters).
* **Optimiser:** plain mini-batch SGD, $\theta \leftarrow \theta - \eta\,\nabla_\theta L$, with $\eta = 0.01$, batch size 128, 30 epochs, and a fresh shuffle every epoch.

Everything in the loop, including the update, uses the manual forward/backward from 1(a).

In [ ]:
@torch.no_grad()   # not needed (no autograd tensors), but makes it explicit that no graph is built
def evaluate(mlp, X, y, batch=5000):
    loss_sum, correct = 0.0, 0
    for i in range(0, len(X), batch):
        logits = mlp.forward(X[i:i + batch])
        loss_sum += mlp.loss_fn.forward(logits, y[i:i + batch]).item() * len(logits)
        correct += (logits.argmax(1) == y[i:i + batch]).sum().item()
    return loss_sum / len(X), correct / len(X)


def train_sgd(mlp, epochs=30, lr=0.01, batch_size=128, seed=SEED):
    gen = torch.Generator().manual_seed(seed)
    hist = {k: [] for k in ['iter_loss', 'train_loss', 'train_acc', 'val_loss', 'val_acc']}
    n = len(Xtr)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        order = torch.randperm(n, generator=gen).to(device)
        run_loss, run_correct = 0.0, 0
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            xb, yb = Xtr[idx], ytr[idx]
            loss, logits, _ = mlp.loss_and_grads(xb, yb)
            for p, g in mlp.params_and_grads():   # SGD step
                p -= lr * g
            hist['iter_loss'].append(loss.item())
            run_loss += loss.item() * len(idx)
            run_correct += (logits.argmax(1) == yb).sum().item()
        vl, va = evaluate(mlp, Xva, yva)
        hist['train_loss'].append(run_loss / n); hist['train_acc'].append(run_correct / n)
        hist['val_loss'].append(vl);             hist['val_acc'].append(va)
        print(f'epoch {ep:2d} | train loss {run_loss / n:.4f} acc {run_correct / n:.4f} '
              f'| val loss {vl:.4f} acc {va:.4f} | {time.time() - t0:.1f}s')
    return hist


set_seed()
mlp = MLP([3072, 512, 256, 10], activation='relu', init='kaiming', device=device)
print('parameters:', mlp.num_params())
hist = train_sgd(mlp, epochs=30, lr=0.01, batch_size=128)

In [ ]:
iters_per_epoch = len(hist['iter_loss']) // len(hist['train_loss'])
epochs = np.arange(1, len(hist['train_loss']) + 1)
it_ep = np.arange(1, len(hist['iter_loss']) + 1) / iters_per_epoch
smooth = np.convolve(hist['iter_loss'], np.ones(50) / 50, mode='valid')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(it_ep, hist['iter_loss'], color='C0', alpha=0.15, lw=0.5, label='train loss (per mini-batch)')
ax[0].plot(it_ep[49:], smooth, color='C0', lw=1.2, label='train loss (50-batch moving avg.)')
ax[0].plot(epochs, hist['val_loss'], 'o-', color='C3', ms=3, label='validation loss (end of epoch)')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('cross-entropy loss'); ax[0].set_title('Loss')
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(epochs, np.array(hist['train_acc']) * 100, 'o-', ms=3, label='train')
ax[1].plot(epochs, np.array(hist['val_acc']) * 100, 'o-', ms=3, color='C3', label='validation')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy (%)'); ax[1].set_title('Accuracy')
ax[1].legend(); ax[1].grid(alpha=0.3)
fig.suptitle('Task 1(c): manual MLP 3072-512-256-10, plain SGD (lr 0.01, batch 128)')
plt.tight_layout(); plt.show()

best = int(np.argmax(hist['val_acc']))
print(f'final validation accuracy : {hist["val_acc"][-1] * 100:.2f}%  (epoch {len(epochs)})')
print(f'best  validation accuracy : {hist["val_acc"][best] * 100:.2f}%  (epoch {best + 1})')
print(f'lowest validation loss    : {min(hist["val_loss"]):.4f}  (epoch {int(np.argmin(hist["val_loss"])) + 1})')
print(f'final train accuracy      : {hist["train_acc"][-1] * 100:.2f}%')

**Interpretation.** _(Fill in the blanks from the printed output above, and adjust any statement that your plot does not support.)_

* **The loss decreases, so the model learns.** The training loss starts near 2.3 ($=\ln 10$, a uniform guess over 10 classes) and falls steadily. This confirms that the manual backward pass and the SGD update work together. The per-batch loss is noisy because each step sees only 128 images. The moving average shows the trend.
* **Validation accuracy** reaches **__%** (best **__%** at epoch __), far above the 10% chance level. MLPs on raw CIFAR-10 pixels typically reach about 50–55%. Flattening the image discards spatial structure. The network has no translation invariance and no weight sharing, so it must learn every pattern separately at every position. This is the motivation for the CNNs in Tasks 2–4.
* **Overfitting.** The training loss keeps falling, but the validation loss reaches its minimum at epoch __ and then rises. Final training accuracy (**__%**) is well above validation accuracy, which plateaus. With 1.7 M parameters and no regularisation, the MLP starts memorising the 45k training images. Early stopping at the validation-loss minimum, weight decay, or data augmentation would help (Task 3).

## Task 1(d): Per-layer gradient norms of a 10-layer MLP at initialisation

The network has 10 Linear layers (3072 → 256 → … → 256 → 10, with 9 activations). It is built with sigmoid and with ReLU activations. Each version is initialised, given one forward/backward pass on a batch of 1 000 training images, and nothing more. The plots show:

* **Left:** $\lVert \partial L/\partial W_l \rVert_F$ for each layer $l$. This is the quantity SGD uses to update $W_l$.
* **Right:** $\lVert \partial L/\partial z_l \rVert_F$, the backpropagated error signal at each layer's pre-activation $z_l$.

Two initialisations are shown. The main comparison uses each activation's standard initialisation: **sigmoid + Xavier** and **ReLU + He**. The swapped pairs (sigmoid + He and ReLU + Xavier) show which effects come from the activation and which come from the initialisation. Each curve is averaged over 5 random initialisations; the shaded band shows the minimum–maximum range.

In [ ]:
def grad_norms_at_init(activation, init, seed, depth=10, width=256, n=1000):
    set_seed(seed)
    mlp = MLP([3072] + [width] * (depth - 1) + [10], activation=activation, init=init, device=device)
    x, y = Xtr[:n], ytr[:n]
    mlp.loss_fn.forward(mlp.forward(x), y)
    grad = mlp.loss_fn.backward()
    delta_norms = []
    for layer in reversed(mlp.layers):            # manual backward, recording dL/dz at each Linear output
        if isinstance(layer, Linear):
            delta_norms.append(grad.norm().item())
        grad = layer.backward(grad)
    w_norms = [l.dW.norm().item() for l in mlp.linears]
    return np.array(w_norms), np.array(delta_norms[::-1])


configs = [('sigmoid', 'xavier', 'C0', '-'), ('relu', 'kaiming', 'C3', '-'),
           ('sigmoid', 'kaiming', 'C0', '--'), ('relu', 'xavier', 'C3', '--')]
results = {}
for act, init, _, _ in configs:
    runs = [grad_norms_at_init(act, init, seed) for seed in range(5)]
    results[(act, init)] = (np.stack([r[0] for r in runs]), np.stack([r[1] for r in runs]))

layers = np.arange(1, 11)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for act, init, c, ls in configs:
    for k, title in enumerate([r'$\|\partial L/\partial W_l\|_F$', r'$\|\partial L/\partial z_l\|_F$']):
        vals = results[(act, init)][k]
        ax[k].plot(layers, vals.mean(0), ls, color=c, marker='o', ms=4,
                   lw=2 if ls == '-' else 1.2, label=f'{act} + {init}')
        ax[k].fill_between(layers, vals.min(0), vals.max(0), color=c, alpha=0.12)
        ax[k].set_title(title); ax[k].set_yscale('log'); ax[k].set_xticks(layers)
        ax[k].set_xlabel('layer $l$ (1 = closest to input)'); ax[k].grid(alpha=0.3, which='both')
ax[0].set_ylabel('gradient norm (log scale)')
ax[0].legend()
fig.suptitle('Task 1(d): 10-layer MLP, per-layer gradient norms at initialisation (mean of 5 seeds)')
plt.tight_layout(); plt.show()

print(f'{"config":>20} | {"||dW_1||":>10} {"||dW_10||":>10} {"ratio 1/10":>11} | {"||dz_1||":>10} {"||dz_10||":>10} {"ratio 1/10":>11}')
for act, init, _, _ in configs:
    w, d = (v.mean(0) for v in results[(act, init)])
    print(f'{act + " + " + init:>20} | {w[0]:10.2e} {w[-1]:10.2e} {w[0] / w[-1]:11.2e} | {d[0]:10.2e} {d[-1]:10.2e} {d[0] / d[-1]:11.2e}')

**Interpretation.** _(Compare these expected trends with your plot and the printed ratios, and quote your own numbers.)_

Backpropagation through one hidden layer computes $\delta_{l} = (\delta_{l+1} W_{l+1}^\top) \odot f'(z_{l})$, and the weight gradient is $\partial L/\partial W_l = h_{l-1}^\top \delta_l$. With $n$ units per layer, each layer multiplies the variance of $\delta$ by roughly

$$
n \,\mathrm{Var}(W)\, \mathbb{E}[f'(z)^2].
$$

* **Sigmoid: vanishing gradients.** $\sigma'(z) = \sigma(z)(1-\sigma(z)) \le 0.25$, and $\sigma'(z) \approx 0.2$–$0.25$ near $z=0$. With Xavier initialisation, $n\,\mathrm{Var}(W) \approx 1$, so each layer scales the backward signal by about 0.2–0.25. The signal shrinks by roughly $0.25^{9} \approx 4\times10^{-6}$ across the 9 sigmoid layers. On the log plot this appears as a straight line falling toward the input. The first layers therefore get gradients several orders of magnitude smaller than the last layer, so they barely change under SGD at any learning rate that is stable for the last layer. Sigmoid outputs are also always positive (mean about 0.5), so $h_{l-1}^\top\delta_l$ has strongly correlated entries, which is a further disadvantage. He initialisation (dashed blue) doubles $\mathrm{Var}(W)$ but only slows the decay: the factor $\mathbb{E}[\sigma'^2] \le 1/16$ is too small for any reasonable initialisation to compensate.
* **ReLU: nearly flat profile.** $\mathrm{ReLU}'(z) \in \{0, 1\}$ and is 1 for about half the units, so $\mathbb{E}[f'^2] = 1/2$. He initialisation sets $\mathrm{Var}(W) = 2/n$, which makes the factor $n \cdot \frac{2}{n} \cdot \frac12 = 1$. The gradient norm then stays roughly constant across all 10 layers, and every layer learns at a similar rate. This is exactly the purpose of He initialisation. The first layer's $\lVert\partial L/\partial W_1\rVert$ differs from the rest mainly because its input is the 3072-dimensional normalised image rather than a 256-dimensional hidden activation.
* **ReLU with Xavier initialisation** (dashed red) has a factor of about $1/2$ in variance, or about $1/\sqrt2$ in norm, per layer. This gives mild vanishing of about $0.7^{9} \approx 0.04$ over the network. Even with ReLU the initialisation has to match the activation, but the effect is far milder than with sigmoid.

**Conclusion.** Sigmoid networks suffer from vanishing gradients because $\sigma' \le 1/4$ is multiplied once per layer. ReLU with He initialisation preserves the gradient scale, which is why deep networks use ReLU-type activations with variance-preserving initialisation. Batch norm and residual connections (Task 3) address the same problem further.